# 12. Text Analysis and Generation

In [5]:
from os.path import basename, exists

# 12. Text Analysis and Generation

# we have covered Python’s core data structures – lists, dictionaries, and tuples –
# and some algorithms that use them. In this chapter, we’ll use them to explore text analysis and Markov generation:
#
# Text analysis is a way to describe the statistical relationships between the words in a document,
# like the probability that one word is followed by another, and

#  Markov generation is a way to generate new text with words and phrases similar to the original text.

## 12.1. Unique words

In [6]:
# 12.1. Unique words

# As a first step toward text analysis, let's read a book -- *The Strange Case Of Dr. Jekyll And Mr. Hyde*
# by Robert Louis Stevenson -- and count the number of unique words.


# def download(url):
#     filename = basename(url)
#     if not exists(filename):
#         from urllib.request import urlretrieve
#
#         local, _ = urlretrieve(url, filename)
#         print("Downloaded " + str(local))
#     return filename
#
#
# # The following cell downloads the book from Project Gutenberg.
# download('https://www.gutenberg.org/cache/epub/43/pg43.txt');
#
# # The version available from Project Gutenberg includes information about the book
# # at the beginning and license information at the end.
# # We'll use `clean_file` from Chapter 8 to remove this material
# # and write a "clean" file that contains only the text of the book.
#
#
# def is_special_line(line):
#     return line.strip().startswith('*** ')
#
#
# def clean_file(input_file, output_file):
#     reader = open(input_file)
#     writer = open(output_file, 'w')
#
#     for line in reader:
#         if is_special_line(line):
#             break
#
#     for line in reader:
#         if is_special_line(line):
#             break
#         writer.write(line)
#
#     reader.close()
#     writer.close()

In [7]:
# file has been created
filename = 'dr_jekyll.txt'

# clean_file('pg43.txt', filename)

unique_words = {}

with open(filename) as reader:
    for line in reader:
        seq = line.split()
        for word in seq:
            unique_words[word] = 1  # If word is already a key in the dictionary,
            # Python updates its existing value to 1. It does not create a duplicate key.

print(len(unique_words))

print(sorted(unique_words, key=len)[-5:])

6039
['chocolate-coloured', 'superiors—behold!”', 'coolness—frightened', 'gentleman—something', 'pocket-handkerchief.']


## 12.2. Punctuation

In [8]:
# 12.2. Punctuation


def split_line(line):  # replace DASH with a space, then use split to remove punctuation
    return line.replace('—', ' ').split()


print(split_line('from 3 - 5,'))  # it leaves hyphenated words alone.

print(split_line('coolness—frightened'))

# Characters in Python strings are in Unicode, which is an international standard used to represent letters
# in nearly every alphabet, numbers, symbols, punctuation marks, and more. The unicodedata module provides
# a category function we can use to tell which characters are punctuation.

import unicodedata

print(unicodedata.category('A'))  # Lu - L letter, u uppercase

print(unicodedata.category('a'))  # Ll

print(unicodedata.category('.'))  # Po - P punctuation, o other subcategory

['from', '3', '-', '5,']
['coolness', 'frightened']
Lu
Ll
Po


In [9]:
# We can find the punctuation marks in the book by checking for characters with categories that begin with 'P'.
# The following loop stores the unique punctuation marks in a dictionary.

punc_marks = {}
with open (filename, encoding='utf-8') as reader:
    for line in reader:
        for char in line:
            category = unicodedata.category(char)
            if category.startswith('P'):
                punc_marks[char] = 1

# To make a list of punctuation marks, we can join the keys of the dictionary into a string.
punctuation = ''.join(punc_marks)
print(punctuation)

# Now that we know which characters in the book are punctuation, we can write a function that takes a word,
# strips punctuation from the beginning and end, and converts it to lower case.


def clean_word(word):  # takes a word, strips punctuation from the beginning and end, and converts it to lower case.
    return word.strip(punctuation).lower()

# print(clean_word('“Behold!”'))  # behold
print(clean_word('pocket-handkerchief'))  # it leaves hyphenated word alone

# Now here’s a loop that uses split_line and clean_word to identify the unique words in the book.
unique_words2 = {}
with open (filename, encoding='utf-8') as reader:
    for line in reader:
        for word in split_line(line):
            word = clean_word(word)
            unique_words2[word] = 1

print(len(unique_words2))

# there are about 4000 unique words. And we can confirm that the list of longest words has been cleaned up.

print(sorted(unique_words2, key=len)[-5:])

# Now let’s see how many times each word is used.

.’;,-“”:?—‘!()_
pocket-handkerchief
4003
['circumscription', 'unimpressionable', 'fellow-creatures', 'chocolate-coloured', 'pocket-handkerchief']


## 12.3. Word frequencies

In [10]:
# 12.3. Word frequencies

# The following loop computes the frequency of each unique word.
word_counter = {}
with open (filename, encoding='utf-8') as reader:
    for line in reader:
        for word in split_line(line):
            word = clean_word(word)
            if word not in word_counter:  # The first time we see a word, we initialize its frequency to 1. If we see the same word again later, we increment its frequency.
                word_counter[word] = 1
            else:
                word_counter[word] += 1


def second_element(t):  # select the second element
    return t[1]

In [11]:

items = sorted(word_counter.items(), key=second_element, reverse=True)
# key=second_element means the items will be sorted according to the frequencies of the words.
# reverse=True means the items will be sorted in reverse order, with the most frequent words first.

# Here are the five most frequent words.
# for word, freq in items[:5]:
#     print(freq, word, sep='\t')  # separated by a tab

# an f-string with a fixed-width field, this looks more neat
for word, freq in items[:5]:
    print(f'{freq:<6}{word}')
# < aligns the frequency to the left.
# 6 reserves six character positions for the frequency.
# word follows immediately after that field.

# In the next section, we’ll encapsulate this loop in a function. And we’ll use it to demonstrate a new feature – optional parameters.

1614  the
972   and
941   of
640   to
640   i


## 12.4. Optional parameters

In [12]:
# 12.4. Optional parameters

#  ndigits  indicates how many decimal places to keep.
print(round(3.141592653589793, ndigits=3))

# But it’s not just built-in functions – we can write functions with optional parameters, too.
# For example, the following function takes two parameters, word_counter and num.


def print_most_common(word_counter, num=5):  # The second parameter looks like an assignment statement,
    # but it’s not – it’s an optional parameter.
    """Print the most common words in word_counter sorted by frequency."""
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(f'{freq:<6}{word}')


print_most_common(word_counter)  # call this function with one argument, num gets the default value, which is 5.

print_most_common(word_counter, 3)  # call this function with two arguments, the second argument gets assigned to num instead of the default value.
# In that case, we would say the optional argument overrides the default value.

# if a function has both required and optional parameters, all of the required parameters have to come first, followed by the optional ones.

3.142
1614  the
972   and
941   of
640   to
640   i
1614  the
972   and
941   of


## 12.5. Dictionary subtraction

In [13]:
# 12.5. Dictionary subtraction

# Suppose we want to spell-check a book – that is, find a list of words that might be misspelled.
# One way to do that is to find words in the book that don’t appear in a list of valid words.

# In previous chapters, we’ve used a list of words that are considered valid in word games like Scrabble.

# we can read the contents of words.txt and split it into a list of strings.
with open ('../words.txt') as reader:
    word_list = reader.read().split()

valid_words = {}
for word in word_list:
    valid_words[word] = 1


In [14]:
# Now, to identify words that appear in the book but not in the word list, we’ll use subtract,
# which takes two dictionaries as parameters and returns a new dictionary
# that contains all the keys from one that are not in the other.


def subtract(d1, d2):
    res = {}
    for key in d1:
        if key not in d2:
            res[key] = d1[key]
    return res


diff = subtract(word_counter, valid_words)

print()
print_most_common(diff)
# The most common “misspelled” words are mostly names and a few single-letter words (Mr. Utterson is Dr. Jekyll’s friend and lawyer).



640   i
628   a
128   utterson
124   mr
98    hyde


In [15]:
# If we select words that only appear once, they are more likely to be actual misspellings.
# We can do that by looping through the items and making a list of words with frequency 1.
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

print(singletons[-5:])  # In Python, dictionaries preserve insertion order in Python 3.7 and later.
print(singletons[: 5])

['gesticulated', 'abjection', 'circumscription', 'reindue', 'fearstruck']
['robert', 'stevenson', 'discourse', 'after-dinner', 'cain’s']


## 12.6. Random numbers

In [16]:
# 12.6. Random numbers

# As a step toward Markov text generation, next we’ll choose a random sequence of words from word_counter.
# But first let’s talk about randomness.

# Given the same inputs, most computer programs are deterministic, which means they generate the same outputs every time.
# Making a program truly nondeterministic turns out to be difficult, but there are ways to fake it.
# One is to use algorithms that generate pseudorandom numbers

# Pseudorandom numbers are not truly random because they are generated by a deterministic computation,
# but just by looking at the numbers it is all but impossible to distinguish them from random.

# The random module provides functions that generate pseudorandom numbers –
# which I will simply call “random” from here on. We can import it like this.

In [17]:

import random

# The random module provides a function called choice that chooses an element from a list at random,
# with every element having the same probability of being chosen
t = [1, 2, 3]
print(random.choice(t))

# If you use choice with a dictionary, you get a KeyError.
# random.choice(word_counter)

2


In [18]:
random.choice(t)  # If you call the function again, you might get the same element again, or a different one.

2

In [19]:
# To choose a random key, you have to put the keys in a list and then call choice.
words = list(word_counter)
print(random.choice(words))

swept


If we generate a random sequence of words, it doesn’t make much sense.

In [20]:
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

countenance befell state mighty potently alter 

Part of the problem is that we are not taking into account that some words are more common than others. The results will be better if we choose words with different “weights”, so that some are chosen more often than others.

If we use the values from word_counter as weights, each word is chosen with a probability that depends on its frequency.

In [21]:
weights = word_counter.values()

The random module provides another function called choices that takes weights as an optional argument.

In [22]:
random.choices(words, weights=weights)

['had']

And it takes another optional argument, k, that specifies the number of words to select.

In [23]:
random_words = random.choices(words, weights=weights, k=6)
random_words

['a', 'when', 'many', 'off', 'be', 'carrying']

The result is a list of strings that we can join into something that’s looks more like a sentence.

In [24]:
' '.join(random_words)

'a when many off be carrying'

If you choose words from the book at random, you get a sense of the vocabulary, but a series of random words seldom makes sense because there is no relationship between successive words.

For example, in a real sentence you expect an article like “the” to be followed by an adjective or a noun, and probably not a verb or adverb. So the next step is to look at these relationships between words.

## 12.7. Bigrams

Instead of looking at one word at a time, now we’ll look at sequences of two words, which are called bigrams. A sequence of three words is called a trigram, and a sequence with some unspecified number of words is called an n-gram.

Let’s write a program that finds all of the bigrams in the book and the number of times each one appears. To store the results, we’ll use a dictionary where

The keys are tuples of strings that represent bigrams, and

The values are integers that represent frequencies.

Let’s call it bigram_counter.

In [25]:
bigram_counter = {}

The following function takes a list of two strings as a parameter. First it makes a tuple of the two strings, which can be used as a key in a dictionary. Then it adds the key to bigram_counter, if it doesn’t exist, or increments the frequency if it does.

In [26]:
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[key] = 1
    else:
        bigram_counter[key] += 1

As we go through the book, we have to keep track of each pair of consecutive words. So if we see the sequence “man is not truly one”, we would add the bigrams “man is”, “is not”, “not truly”, and so on.

To keep track of these bigrams, we’ll use a list called window, because it is like a window that slides over the pages of the book, showing only two words at a time. Initially, window is empty.

In [27]:
window = []

def process_word(word):  # use this function to process the words one at a time.
    window.append(word)

    if len(window) == 2:
        count_bigram(window)
        window.pop(0)


# The first time this function is called, it appends the given word to window. Since there is only one word in the window, we don’t have a bigram yet, so the function ends.

#The second time it’s called – and every time thereafter – it appends a second word to window. Since there are two words in the window, it calls count_bigram to keep track of how many times each bigram appears. Then it uses pop to remove the first word from the window.

In [28]:
# The following program loops through the words in the book and processes them one at a time.

for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word(word)

In [29]:
# The result is a dictionary that maps from each bigram to the number of times it appears. We can use print_most_common to see the most common bigrams.
print_most_common(bigram_counter)

178   ('of', 'the')
139   ('in', 'the')
94    ('it', 'was')
80    ('and', 'the')
73    ('to', 'the')


In [30]:
# Looking at these results, we can get a sense of which pairs of words are most likely to appear together. We can also use the results to generate random text, like this.

bigrams = list(bigram_counter)
weights = bigram_counter.values()
random_bigrams = random.choices(bigrams, weights=weights, k=6)

In [31]:
# bigrams is a list of the bigrams that appear in the books. weights is a list of their frequencies, so random_bigrams is a sample where the probability a bigram is selected is proportional to its frequency.

# Here are the results.

for pair in random_bigrams:
    print(' '.join(pair), end=' ')

a considerable he had this brief an innocent was in but from 

In [ ]:
# This way of This way of generating text is better than choosing random words, but still doesn’t make a lot of sense.

# 12.8. Markov analysis

In [33]:
# We can do better with Markov chain text analysis, which computes, for each word in a text, the list of words that come next. As an example, we’ll analyze these lyrics from the Monty Python song Eric, the Half a Bee:

song = """
Half a bee, philosophically,
Must, ipso facto, half not be.
But half the bee has got to be
Vis a vis, its entity. D'you see?
"""

In [34]:
# To store the results, we’ll use a dictionary that maps from each word to the list of words that follow it.
successor_map = {}

# As an example, let’s start with the first two words of the song.
first = 'half'
second = 'a'

# If the first word is not in successor_map, we have to add a new item that maps from the first word to a list containing the second word.
successor_map[first] = [second]
successor_map

{'half': ['a']}

In [35]:
# If the first word is already in the dictionary, we can look it up to get the list of successors we’ve seen so far, and append the new one.

first = 'half'
second = 'not'

successor_map[first].append(second)
successor_map

{'half': ['a', 'not']}

In [36]:
# The following function encapsulates these steps.


def add_bigram(bigram):
    first, second = bigram

    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

In [37]:
# If the same bigram appears more that once, the second word is added to the list more than once. In this way, successor_map keeps track of how many times each successor appears.

# As we did in the previous section, we’ll use a list called window to store pairs of consecutive words. And we’ll use the following function to process the words one at a time.


def process_word_bigram(word):
    window.append(word)

    if len(window) == 2:
        add_bigram(window)
        window.pop(0)

In [38]:
# Here’s how we use it to process the words in the song.
successor_map = {}
window = []

for word in song.split():
    word = clean_word(word)
    process_word_bigram(word)

In [39]:
# And here are the results.
successor_map

{'half': ['a', 'not', 'the'],
 'a': ['bee', 'vis'],
 'bee': ['philosophically', 'has'],
 'philosophically': ['must'],
 'must': ['ipso'],
 'ipso': ['facto'],
 'facto': ['half'],
 'not': ['be'],
 'be': ['but', 'vis'],
 'but': ['half'],
 'the': ['bee'],
 'has': ['got'],
 'got': ['to'],
 'to': ['be'],
 'vis': ['a', 'its'],
 'its': ['entity'],
 'entity': ["d'you"],
 "d'you": ['see']}

In [40]:
# The word 'half' can be followed by 'a', 'not', or 'the'. The word 'a' can be followed by 'bee' or 'vis'. Most of the other words appear only once, so they are followed by only a single word.

# Now let’s analyze the book.
successor_map = {}
window = []

for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word_bigram(word)

# We can look up any word and find the words that can follow it.
successor_map['going']

['east', 'in', 'to', 'to', 'up', 'to', 'of']

In [ ]:
# In this list of successors, notice that the word 'to' appears three times – the other successors only appear once.

# 12.9. Generating text

In [41]:
# We can use the results from the previous section to generate new text with the same relationships between consecutive words as in the original. Here’s how it works:

# Starting with any word that appears in the text, we look up its possible successors and choose one at random.

# Then, using the chosen word, we look up its possible successors, and choose one at random.

# We can repeat this process to generate as many words as we want. As an example, let’s start with the word 'although'. Here are the words that can follow it.

word = 'although'
successors = successor_map[word]
successors

['i', 'a', 'it', 'the', 'we', 'they', 'i']

In [42]:
# We can use choice to choose from the list with equal probability.
word = random.choice(successors)
word

'it'

In [43]:
# If the same word appears more than once in the list, it is more likely to be selected.

# Repeating these steps, we can use the following loop to generate a longer series.

for i in range(10):
    successors = successor_map[word]
    word = random.choice(successors)
    print(word, end=' ')

was done tearing him but of good for any particular 

In [44]:
# The result sounds more like a real sentence, but it still doesn’t make much sense.

# We can do better using more than one word as a key in successor_map. For example, we can make a dictionary that maps from each bigram – or trigram – to the list of words that come next. As an exercise, you’ll have a chance to implement this analysis and see what the results look like.

# 12.10. Debugging

In [45]:
"""
At this point we are writing more substantial programs, and you might find that you are spending more time debugging. If you are stuck on a difficult bug, here are a few things to try:

Reading: Examine your code, read it back to yourself, and check that it says what you meant to say.

Running: Experiment by making changes and running different versions. Often if you display the right thing at the right place in the program, the problem becomes obvious, but sometimes you have to build scaffolding.

Ruminating: Take some time to think! What kind of error is it: syntax, runtime, or semantic? What information can you get from the error messages, or from the output of the program? What kind of error could cause the problem you’re seeing? What did you change last, before the problem appeared?

Rubberducking: If you explain the problem to someone else, you sometimes find the answer before you finish asking the question. Often you don’t need the other person; you could just talk to a rubber duck. And that’s the origin of the well-known strategy called rubber duck debugging. I am not making this up – see https://en.wikipedia.org/wiki/Rubber_duck_debugging.

Retreating: At some point, the best thing to do is back up – undoing recent changes – until you get to a program that works. Then you can start rebuilding.

Resting: If you give your brain a break, sometime it will find the problem for you.

Beginning programmers sometimes get stuck on one of these activities and forget the others. Each activity comes with its own failure mode.

For example, reading your code works if the problem is a typographical error, but not if the problem is a conceptual misunderstanding. If you don’t understand what your program does, you can read it 100 times and never see the error, because the error is in your head.

Running experiments can work, especially if you run small, simple tests. But if you run experiments without thinking or reading your code, it can take a long time to figure out what’s happening.

You have to take time to think. Debugging is like an experimental science. You should have at least one hypothesis about what the problem is. If there are two or more possibilities, try to think of a test that would eliminate one of them.

But even the best debugging techniques will fail if there are too many errors, or if the code you are trying to fix is too big and complicated. Sometimes the best option is to retreat, simplifying the program until you get back to something that works.

Beginning programmers are often reluctant to retreat because they can’t stand to delete a line of code (even if it’s wrong). If it makes you feel better, copy your program into another file before you start stripping it down. Then you can copy the pieces back one at a time.

Finding a hard bug requires reading, running, ruminating, retreating, and sometimes resting. If you get stuck on one of these activities, try the others.
"""

'\nAt this point we are writing more substantial programs, and you might find that you are spending more time debugging. If you are stuck on a difficult bug, here are a few things to try:\n\nReading: Examine your code, read it back to yourself, and check that it says what you meant to say.\n\nRunning: Experiment by making changes and running different versions. Often if you display the right thing at the right place in the program, the problem becomes obvious, but sometimes you have to build scaffolding.\n\nRuminating: Take some time to think! What kind of error is it: syntax, runtime, or semantic? What information can you get from the error messages, or from the output of the program? What kind of error could cause the problem you’re seeing? What did you change last, before the problem appeared?\n\nRubberducking: If you explain the problem to someone else, you sometimes find the answer before you finish asking the question. Often you don’t need the other person; you could just talk to

# 12.11. Glossary

In [46]:
"""
default value: The value assigned to a parameter if no argument is provided.

override: To replace a default value with an argument.

deterministic: A deterministic program does the same thing each time it runs, given the same inputs.

pseudorandom: A pseudorandom sequence of numbers appears to be random, but is generated by a deterministic program.

bigram: A sequence of two elements, often words.

trigram: A sequence of three elements.

n-gram: A sequence of an unspecified number of elements.

rubber duck debugging: A way of debugging by explaining a problem aloud to an inanimate object.
"""

'\ndefault value: The value assigned to a parameter if no argument is provided.\n\noverride: To replace a default value with an argument.\n\ndeterministic: A deterministic program does the same thing each time it runs, given the same inputs.\n\npseudorandom: A pseudorandom sequence of numbers appears to be random, but is generated by a deterministic program.\n\nbigram: A sequence of two elements, often words.\n\ntrigram: A sequence of three elements.\n\nn-gram: A sequence of an unspecified number of elements.\n\nrubber duck debugging: A way of debugging by explaining a problem aloud to an inanimate object.\n'

## 12.12. Exercises

In [47]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

Exception reporting mode: Verbose


### 12.12.1. Ask a virtual assistant

In [48]:
# In add_bigram, the if statement creates a new list or appends an element to an existing list, depending on whether the key is already in the dictionary.


def add_bigram(bigram):
    first, second = bigram

    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

In [49]:
# Dictionaries provide a method called setdefault that we can use to do the same thing more concisely. Ask a virtual assistant how it works,
#
# setdefault() checks whether a key exists in a dictionary.
# - If the key already exists, it returns the existing value without changing it.
# - If the key doesn't exist, it inserts the key with a default value and returns that value.
# the syntax is : dictionary.setdefault(key, default_value)

# or copy add_bigram into a virtual assistant and ask “Can you rewrite this using setdefault?”


def add_bigram(bigram):
    first, second = bigram
    successor_map.setdefault(first, []).append(second)


# setdefault() creates an empty list as the value when the key doesn't exist. Then .append(second) adds second to that list.


In [50]:
# In this chapter we implemented Markov chain text analysis and generation. If you are curious, you can ask a virtual assistant for more information on the topic. One of the things you might learn is that virtual assistants use algorithms that are similar in many ways – but also different in important ways. Ask a VA, “What are the differences between large language models like GPT and Markov chain text analysis?”

"""
1. How do they generate text?

A. Markov chain text generation

A Markov chain learns which words tend to follow other words by counting patterns in a text corpus. It uses those observed transitions to choose the next word.
For example, if the word "the" is followed by "cat" 60 times and "dog" 40 times, the model might choose "cat" with probability 60% and "dog" with probability 40%.

Imagine these bigrams:

successor_map = {
    "the": ["cat", "dog"],
    "cat": ["sat", "slept"],
    "dog": ["ran"],
    "sat": ["down"],
}
If the current word is "the", your program can randomly select "cat" or "dog".
If it selects "cat", it can then select "sat" or "slept". A possible output is:
the cat sat down

But consider what happens when the model generates "the dog" and then needs to continue. If "dog" has only been observed followed by "ran", the next word must be "ran" under this map.
The model follows the transitions it has recorded. It doesn't inherently know that a cat can sleep, a dog can bark, or that a sentence should express a coherent idea.

B. Large language model (LLM)

An LLM learns statistical patterns from enormous amounts of text using a neural network. It predicts the next token based on the preceding context, using learned representations of language, concepts, and relationships.
It can use much more context than a simple bigram model, allowing it to produce explanations, write code, translate languages, and follow instructions.

An LLM, by contrast, can draw on more complex learned relationships and a broader context. It might generate:
The dog ran across the garden to chase a ball.

It can produce this sentence even if that exact sequence never appeared in its training data.
"""

'\n1. How do they generate text?\n\nA. Markov chain text generation\n\nA Markov chain learns which words tend to follow other words by counting patterns in a text corpus. It uses those observed transitions to choose the next word.\nFor example, if the word "the" is followed by "cat" 60 times and "dog" 40 times, the model might choose "cat" with probability 60% and "dog" with probability 40%.\n\nImagine these bigrams:\n\nsuccessor_map = {\n    "the": ["cat", "dog"],\n    "cat": ["sat", "slept"],\n    "dog": ["ran"],\n    "sat": ["down"],\n}\nIf the current word is "the", your program can randomly select "cat" or "dog".\nIf it selects "cat", it can then select "sat" or "slept". A possible output is:\nthe cat sat down\n\nBut consider what happens when the model generates "the dog" and then needs to continue. If "dog" has only been observed followed by "ran", the next word must be "ran" under this map.\nThe model follows the transitions it has recorded. It doesn\'t inherently know that a c

### 12.12.2. Exercise

In [ ]:
"""
Write a function that counts the number of times each trigram (sequence of three words) appears. If you test your function with the text of Dr. Jekyll and Mr. Hyde, you should find that the most common trigram is “said the lawyer”.

Hint: Write a function called count_trigram that is similar to count_bigram. Then write a function called process_word_trigram that is similar to process_word_bigram.
"""
trigram_counter = {}
window = []
successor_map = {}

def count_trigram(trigram):
    key = tuple(trigram)
    if key not in trigram_counter:
        trigram_counter[key] = 1
    else:
        trigram_counter[key] += 1


def add_trigram(trigram):
    first, second, third = trigram

    if first not in successor_map:
        successor_map[first] = [second, third]  # {'dag': [('is', 'barking')]}
    else:
        successor_map[first].append(second, third)


def process_word_trigram(word):
    window.append(word)

    if len(window) == 3:
        add_trigram(window)
        window.pop(0)


for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)
